In [1]:
import json
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)

PROJECT_ROOT = Path("..")

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "players_clean.csv"
)

FEATURE_CONFIG = (
    PROJECT_ROOT
    / "config"
    / "feature_config.json"
)

df = pd.read_csv(DATA_PATH)

with open(FEATURE_CONFIG) as f:
    feature_config = json.load(f)

ATTRIBUTE_MAP = feature_config["attribute_map"]
POSITION_GROUPS = feature_config["position_groups"]

print("Dataset shape:", df.shape)
print("Number of players:", df["Player"].nunique())
print("Attributes:", list(ATTRIBUTE_MAP.keys()))

Dataset shape: (1692, 43)
Number of players: 1691
Attributes: ['Finishing', 'Creativity', 'Passing', 'Progression', 'Defending', 'BallSecurity', 'Aerial']


In [2]:
df.head()

,Player,Nation,Pos,Squad,Comp,Age,Born,Min,CrdY,CrdR,2CrdY,Fls,Gls,xG,Sh,SoT,G/Sh,Ast,xA,KP,SCA,GCA,TB,Cmp%,PrgP,PrgDist,PPA,Sw,PrgC,Carries,CPA,Succ,Tkl,TklW,Int,Blocks_stats_defense,Clr,Recov,Mis,Dis,Rec,Won,Won%
0,Yunis Abdelhamid,ma MAR,DF,Saint-Étienne,fr Ligue 1,36.0,1987.0,1033,2,0,0,16,0,0.2,2,1,0.000000,0,0.3,2,6,0,0,86.732673,22,2404,0,0,4,322,0,3,17,11,14,26,57,42,8,3,393,19,76.000000
1,Himad Abdelli,dz ALG,MF,Angers,fr Ligue 1,24.0,1999.0,2842,4,0,0,48,6,6.4,35,8,0.171429,1,3.7,39,94,5,23,82.117225,207,6031,38,13,107,1183,17,57,48,32,26,24,28,193,55,51,1446,13,28.260870
2,Mohamed Abdelmoneim,eg EGY,DF,Nice,fr Ligue 1,25.0,1999.0,855,2,0,0,9,0,0.0,0,0,NaN,0,0.2,0,8,0,0,90.403071,52,2620,1,7,6,342,0,1,21,14,15,11,56,50,4,3,386,31,77.500000
3,Ali Abdi,tn TUN,DF,Nice,fr Ligue 1,30.0,1993.0,1393,2,0,0,27,5,4.3,29,11,0.172414,2,1.7,20,45,3,0,74.041298,42,2701,21,2,35,391,9,12,43,25,20,20,37,72,32,14,481,22,59.459459
4,Abel,es ESP,DF,Osasuna,es La Liga,23.0,2000.0,2074,3,0,0,25,2,0.5,12,4,0.166667,0,1.5,16,39,4,3,72.455649,77,5046,18,2,50,605,7,18,61,37,18,29,64,106,27,18,683,20,45.454545


In [3]:
# Counting statistics to convert into per-90 values

PER_90_COLUMNS = [
    "Gls", "xG", "Sh", "SoT",
    "Ast", "xA", "KP", "SCA", "GCA", "TB",
    "PrgP", "PrgDist", "PPA", "Sw",
    "PrgC", "Carries", "CPA", "Succ",
    "Tkl", "TklW", "Int",
    "Blocks_stats_defense", "Clr", "Recov",
    "Mis", "Dis", "Rec",
    "Won"
]

# Calculate per-90 statistics
for col in PER_90_COLUMNS:
    if col in df.columns:
        df[f"{col}_per90"] = (
            df[col] / df["Min"] * 90
        )

print("Per-90 features created:", len(PER_90_COLUMNS))

df[
    ["Player", "Min", "Gls", "Gls_per90",
     "Tkl", "Tkl_per90"]
].head()

Per-90 features created: 28


,Player,Min,Gls,Gls_per90,Tkl,Tkl_per90
0,Yunis Abdelhamid,1033,0,0.000000,17,1.481123
1,Himad Abdelli,2842,6,0.190007,48,1.520056
2,Mohamed Abdelmoneim,855,0,0.000000,21,2.210526
3,Ali Abdi,1393,5,0.323044,43,2.778177
4,Abel,2074,2,0.086789,61,2.647059


In [4]:
per90_columns = [
    f"{col}_per90"
    for col in PER_90_COLUMNS
    if f"{col}_per90" in df.columns
]

print("Per-90 columns:", len(per90_columns))

print(
    "Infinite values:",
    np.isinf(df[per90_columns]).sum().sum()
)

print(
    "Missing values:",
    df[per90_columns].isna().sum().sum()
)

print(
    "Negative values:",
    (df[per90_columns] < 0).sum().sum()
)

Per-90 columns: 28
Infinite values: 0
Missing values: 0
Negative values: 0


In [5]:
# Features where lower values indicate better performance

NEGATIVE_FEATURES = [
    "Mis_per90",
    "Dis_per90"
]

# Features where higher values indicate better performance

POSITIVE_FEATURES = [
    f"{col}_per90"
    for col in PER_90_COLUMNS
    if f"{col}_per90" not in NEGATIVE_FEATURES
]

# Percentage and ratio features
RATE_FEATURES = [
    "Cmp%",
    "Won%",
    "G/Sh"
]

print("Positive features:", len(POSITIVE_FEATURES))
print("Negative features:", len(NEGATIVE_FEATURES))
print("Rate features:", len(RATE_FEATURES))

Positive features: 26
Negative features: 2
Rate features: 3


In [6]:
SCORING_FEATURES = (
    POSITIVE_FEATURES
    + NEGATIVE_FEATURES
    + RATE_FEATURES
)

print("Total scoring features:", len(SCORING_FEATURES))

Total scoring features: 31


In [7]:
# Create a separate percentile column for each feature

for col in SCORING_FEATURES:

    percentile_col = f"{col}_pct"

    if col in NEGATIVE_FEATURES:
        df[percentile_col] = (
            df.groupby("Pos")[col]
            .rank(pct=True, ascending=False) * 100
        )

    else:
        df[percentile_col] = (
            df.groupby("Pos")[col]
            .rank(pct=True, ascending=True) * 100
        )

print("Percentile features created:", len(SCORING_FEATURES))

Percentile features created: 31


In [8]:
PERCENTILE_COLUMNS = [
    f"{col}_pct"
    for col in SCORING_FEATURES
]

print("Percentile columns:", len(PERCENTILE_COLUMNS))

print(
    "Minimum percentile:",
    df[PERCENTILE_COLUMNS].min().min()
)

print(
    "Maximum percentile:",
    df[PERCENTILE_COLUMNS].max().max()
)

print(
    "Missing percentile values:",
    df[PERCENTILE_COLUMNS].isna().sum().sum()
)

print(
    "Values outside 0-100:",
    (
        (df[PERCENTILE_COLUMNS] < 0) |
        (df[PERCENTILE_COLUMNS] > 100)
    ).sum().sum()
)

Percentile columns: 31
Minimum percentile: 0.14245014245014245
Maximum percentile: 100.0
Missing percentile values: 8
Values outside 0-100: 0


In [9]:
df[
    [
        "Player", "Pos",
        "Gls_per90", "Gls_per90_pct",
        "Tkl_per90", "Tkl_per90_pct",
        "Mis_per90", "Mis_per90_pct"
    ]
].head(10)

,Player,Pos,Gls_per90,Gls_per90_pct,Tkl_per90,Tkl_per90_pct,Mis_per90,Mis_per90_pct
0,Yunis Abdelhamid,DF,0.000000,24.216524,1.481123,36.752137,0.696999,51.994302
1,Himad Abdelli,MF,0.190007,78.368794,1.520056,30.319149,1.741731,33.333333
2,Mohamed Abdelmoneim,DF,0.000000,24.216524,2.210526,76.210826,0.421053,76.210826
3,Ali Abdi,DF,0.323044,99.715100,2.778177,94.017094,2.067480,2.279202
4,Abel,DF,0.086789,79.914530,2.647059,91.880342,1.171649,23.789174
5,Matthis Abline,FW,0.292630,51.643192,0.357659,8.920188,2.991329,27.230047
6,Abner,DF,0.082569,78.062678,2.394495,83.048433,1.155963,24.216524
7,Zakaria Aboukhlal,MF,0.331405,93.439716,1.041557,10.992908,3.029984,3.723404
8,Abdel Abqar,DF,0.000000,24.216524,1.315469,25.356125,0.657734,54.273504
9,Tammy Abraham,FW,0.228041,38.262911,1.292230,69.718310,1.748311,86.619718


In [10]:
missing_percentiles = (
    df[PERCENTILE_COLUMNS]
    .isna()
    .sum()
)

print(
    missing_percentiles[
        missing_percentiles > 0
    ]
)

G/Sh_pct    8
dtype: int64


In [11]:
for col in PERCENTILE_COLUMNS:

    missing_players = df.loc[
        df[col].isna(),
        ["Player", "Pos", col.replace("_pct", "")]
    ]

    if not missing_players.empty:
        print(f"\nMissing values in {col}:")
        display(missing_players)


Missing values in G/Sh_pct:


,Player,Pos,G/Sh
2,Mohamed Abdelmoneim,DF,NaN
860,Toni Lato,DF,NaN
889,Victor Lindelöf,DF,NaN
947,Jordy Makengo,DF,NaN
1026,Ferland Mendy,DF,NaN
1342,Falaye Sacko,DF,NaN
1391,Joël Schingtienne,DF,NaN
1608,Kevin Vogt,DF,NaN


In [13]:
# Original counting features use their per-90 percentiles.
# Rate features retain their original percentiles.

for attribute, details in ATTRIBUTE_MAP.items():

    feature_percentiles = []

    for feature in details["features"]:

        if feature in RATE_FEATURES:
            percentile_col = f"{feature}_pct"
        else:
            percentile_col = f"{feature}_per90_pct"

        if percentile_col in df.columns:
            feature_percentiles.append(percentile_col)

    df[f"{attribute}_Score"] = (
        df[feature_percentiles].mean(axis=1)
    )

print("Attribute scores created:")
print([
    f"{attribute}_Score"
    for attribute in ATTRIBUTE_MAP
])

Attribute scores created:
['Finishing_Score', 'Creativity_Score', 'Passing_Score', 'Progression_Score', 'Defending_Score', 'BallSecurity_Score', 'Aerial_Score']


In [14]:
ATTRIBUTE_COLUMNS = [
    f"{attribute}_Score"
    for attribute in ATTRIBUTE_MAP
]

print("Attribute columns:", len(ATTRIBUTE_COLUMNS))

print("\nMinimum score:")
print(df[ATTRIBUTE_COLUMNS].min().min())

print("\nMaximum score:")
print(df[ATTRIBUTE_COLUMNS].max().max())

print("\nMissing scores:")
print(df[ATTRIBUTE_COLUMNS].isna().sum())

print(
    "\nScores outside 0-100:",
    (
        (df[ATTRIBUTE_COLUMNS] < 0) |
        (df[ATTRIBUTE_COLUMNS] > 100)
    ).sum().sum()
)

Attribute columns: 7

Minimum score:
0.2136752136752137

Maximum score:
100.0

Missing scores:
Finishing_Score       0
Creativity_Score      0
Passing_Score         0
Progression_Score     0
Defending_Score       0
BallSecurity_Score    0
Aerial_Score          0
dtype: int64

Scores outside 0-100: 0


In [15]:
df[
    ["Player", "Pos"] + ATTRIBUTE_COLUMNS
].head(10).round(2)

,Player,Pos,Finishing_Score,Creativity_Score,Passing_Score,Progression_Score,Defending_Score,BallSecurity_Score,Aerial_Score
0,Yunis Abdelhamid,DF,21.44,19.42,22.10,22.26,61.32,52.85,59.58
1,Himad Abdelli,MF,66.74,56.97,75.59,86.03,44.44,44.95,19.59
2,Mohamed Abdelmoneim,DF,8.30,18.54,60.93,43.53,79.25,67.95,78.42
3,Ali Abdi,DF,96.67,74.20,33.87,61.71,69.75,10.97,42.06
4,Abel,DF,60.75,63.24,44.85,63.90,61.68,15.92,22.40
5,Matthis Abline,FW,50.80,35.80,31.38,49.44,29.62,20.60,80.99
6,Abner,DF,70.94,72.06,71.98,78.83,53.49,40.42,12.43
7,Zakaria Aboukhlal,MF,91.74,34.53,5.66,40.92,10.58,9.04,64.98
8,Abdel Abqar,DF,28.47,40.19,44.78,21.92,46.23,39.60,53.42
9,Tammy Abraham,FW,50.05,67.86,42.59,14.93,54.46,51.82,87.03


In [16]:
df[ATTRIBUTE_COLUMNS].describe().round(2)

,Finishing_Score,Creativity_Score,Passing_Score,Progression_Score,Defending_Score,BallSecurity_Score,Aerial_Score
count,1692.00,1692.00,1692.00,1692.00,1692.00,1692.00,1692.00
mean,50.05,50.09,50.09,50.09,50.09,50.09,50.09
std,23.91,23.33,18.33,20.96,18.78,20.87,27.82
min,1.13,3.72,5.66,0.92,3.33,2.74,0.21
25%,29.34,30.73,36.50,34.74,36.63,34.24,26.56
50%,50.05,48.79,49.28,50.39,50.25,50.59,50.16
75%,68.92,67.92,63.20,65.88,64.07,66.01,74.17
max,98.64,99.69,98.20,97.62,94.80,97.03,100.00


In [17]:
df.groupby("Pos")[ATTRIBUTE_COLUMNS].mean().round(2)

,Finishing_Score,Creativity_Score,Passing_Score,Progression_Score,Defending_Score,BallSecurity_Score,Aerial_Score
Pos,,,,,,,
DF,49.98,50.07,50.07,50.07,50.07,50.07,50.07
FW,50.12,50.12,50.12,50.12,50.12,50.12,50.12
MF,50.09,50.09,50.09,50.09,50.09,50.09,50.09


In [18]:
OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "players_engineered.csv"
)

OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

df.to_csv(OUTPUT_PATH, index=False)

print("Engineered dataset saved!")
print("Location:", OUTPUT_PATH)
print("Shape:", df.shape)

Engineered dataset saved!
Location: ..\data\processed\players_engineered.csv
Shape: (1692, 109)
